1) Parametros (ajusta N_IMU y ODR_HZ a lo que tengas conectado)

In [ ]:
import time, struct
import numpy as np
from pynq import Overlay, MMIO

BIT      = '/home/xilinx/zybo_multi.bit'
N_IMU    = 2                 # IMU conectadas (0..N_IMU-1 = JE, JD, JC, JA segun tu cableado)
ODR_HZ   = 119               # 119 o 952; el bitstream/firmware lo soportan igual que main_pl.c
ODR_CODE = {119: 3, 952: 6}[ODR_HZ]
DT_DIV   = {119: 1.0, 952: 8.0}[ODR_HZ]

NAV_BASES = [0x40000000 + 0x10000 * i for i in range(4)]
MAD_BASE, R2F_BASE = 0x40040000, 0x40050000
NAV_CTRL, NAV_ODR_CFG, NAV_RANGE_CFG, NAV_STATUS, NAV_SAMPLE_CNT, NAV_RAW0 = 0x00, 0x04, 0x08, 0x40, 0x44, 0x50
NAV_OFF_G, NAV_OFF_M, NAV_AXIS_CFG = 0x20, 0x2C, 0x38
NAV_SNAP_SEQ, NAV_SNAP_TS_GDA, NAV_SNAP_TS_END = 0x74, 0x78, 0x7C
HLS_AP_CTRL, AP_START, AP_AUTORESTART = 0x00, 1, 0x80
R2F_FS_G_REG = 0x10
MAD_BETA, MAD_RESET, MAD_Q_OUT, MAD_FRAME_CNT, MAD_DT_IMU = 0x10, 0x20, 0x40, 0x80, 0x90
ST_CONFIG_DONE, ST_ERROR = 1 << 1, 1 << 4
FS_G_SELECTED, AXIS_CFG_PRESET = 1, 0x039
BETA_START, BETA_RUN, BETA_START_MS = 1.0, 0.1, 3000
MAG_PRESET = [[1537, 1372, -2025], [-1952, 2124, -1233], [-2188, 2152, -1273], [1508, 1242, -1704]]
DT_IMU = [8.280e-3 / DT_DIV, 8.307e-3 / DT_DIV, 8.480e-3 / DT_DIV, 8.057e-3 / DT_DIV]

def f2u(f): return struct.unpack('<I', struct.pack('<f', f))[0]
def u2f(u): return struct.unpack('<f', struct.pack('<I', u & 0xFFFFFFFF))[0]
def s16(v): v &= 0xFFFF; return v - 0x10000 if v & 0x8000 else v

2) Cargar el bitstream y mapear los registros

In [ ]:
ol  = Overlay(BIT)                                   # descarga el .bit y ajusta los relojes con el .hwh
nav = [MMIO(NAV_BASES[i], 0x1000) for i in range(4)]
mad = MMIO(MAD_BASE, 0x10000)
r2f = MMIO(R2F_BASE, 0x10000)
print('Overlay cargado:', BIT)

3) Lectura de un cuaternion coherente con frame_cnt

In [ ]:
def read_q(i):
    c1 = mad.read(MAD_FRAME_CNT + 4 * i)
    q = [u2f(mad.read(MAD_Q_OUT + 16 * i + 4 * k)) for k in range(4)]
    c2 = mad.read(MAD_FRAME_CNT + 4 * i)
    return q, c1, c1 == c2          # q = (q0, q1, q2, q3) tal como los entrega el nucleo

def snap(i):
    while True:
        s1 = nav[i].read(NAV_SNAP_SEQ); tg = nav[i].read(NAV_SNAP_TS_GDA); te = nav[i].read(NAV_SNAP_TS_END)
        if nav[i].read(NAV_SNAP_SEQ) == s1: return s1, tg, te

4) Configuracion (igual que main_pl.c)

In [ ]:
def configure():
    for i in range(N_IMU):
        nav[i].write(NAV_ODR_CFG, (7 << 6) | (ODR_CODE << 3) | ODR_CODE)
        nav[i].write(NAV_RANGE_CFG, FS_G_SELECTED & 3)
        nav[i].write(NAV_AXIS_CFG, AXIS_CFG_PRESET)
        for k in range(3):
            nav[i].write(NAV_OFF_G + 4 * k, 0)
            nav[i].write(NAV_OFF_M + 4 * k, MAG_PRESET[i][k] & 0xFFFF)
    r2f.write(R2F_FS_G_REG, FS_G_SELECTED & 3)
    r2f.write(HLS_AP_CTRL, AP_START | AP_AUTORESTART)
    mad.write(MAD_BETA, f2u(BETA_START))
    for i in range(4): mad.write(MAD_DT_IMU + 4 * i, f2u(DT_IMU[i]))
    mad.write(MAD_RESET, (1 << N_IMU) - 1)
    mad.write(HLS_AP_CTRL, AP_START | AP_AUTORESTART)
    for i in range(N_IMU): nav[i].write(NAV_CTRL, 1)           # arranca la adquisicion SPI
    for i in range(N_IMU):
        t0 = time.time()
        while not (nav[i].read(NAV_STATUS) & (ST_CONFIG_DONE | ST_ERROR)):
            if time.time() - t0 > 2: raise RuntimeError('IMU%d: la configuracion no termina (STATUS=0x%08x)' % (i, nav[i].read(NAV_STATUS)))
            time.sleep(0.001)
        if nav[i].read(NAV_STATUS) & ST_ERROR:
            raise RuntimeError('IMU%d: ERROR de configuracion (STATUS=0x%08x)' % (i, nav[i].read(NAV_STATUS)))
    t0 = time.time()
    while not all(mad.read(MAD_FRAME_CNT + 4 * i) for i in range(N_IMU)):
        if time.time() - t0 > 2: raise RuntimeError('el nucleo no recibe tramas de todas las IMU')
        time.sleep(0.001)

def measure_dt(seconds=3):
    s0 = [snap(i) for i in range(N_IMU)]
    time.sleep(seconds)
    for i in range(N_IMU):
        s1 = snap(i); ds = (s1[0] - s0[i][0]) & 0xFFFFFFFF; dc = (s1[1] - s0[i][1]) & 0xFFFFFFFF
        dt = dc / ds * 1e-8 if ds >= 100 else DT_IMU[i]
        mad.write(MAD_DT_IMU + 4 * i, f2u(dt))
        print('IMU%d: periodo medido %.3f us (%.2f Hz)' % (i, dt * 1e6, 1 / dt))

def gyro_bias():
    print('Midiendo el sesgo del giroscopio (~2 s, placas QUIETAS)...')
    for i in range(N_IMU):
        acc = np.zeros(3, np.int64); n = 0; last = nav[i].read(NAV_SAMPLE_CNT)
        while n < 2 * ODR_HZ:
            c = nav[i].read(NAV_SAMPLE_CNT)
            if c == last: continue
            last = c
            acc += [s16(nav[i].read(NAV_RAW0 + 4 * k)) for k in range(3)]; n += 1
        off = (acc // n).tolist()
        for k in range(3): nav[i].write(NAV_OFF_G + 4 * k, off[k] & 0xFFFF)
        print('IMU%d: sesgo giro = %s cuentas' % (i, off))

def start_filter():
    configure(); measure_dt(3); gyro_bias()
    mad.write(MAD_RESET, 0); mad.write(MAD_BETA, f2u(BETA_START))
    time.sleep(BETA_START_MS / 1000); mad.write(MAD_BETA, f2u(BETA_RUN))
    print('Filtro en marcha (beta %.2f)' % BETA_RUN)

5) Arranque (placas quietas durante unos 8 s)

In [ ]:
start_filter()

6) Mostrar cuaterniones una vez por segundo y contar tramas perdidas

In [ ]:
def monitor(seconds=10):
    c_acq0 = [nav[i].read(NAV_SAMPLE_CNT) for i in range(N_IMU)]
    c_core0 = [mad.read(MAD_FRAME_CNT + 4 * i) for i in range(N_IMU)]
    t0 = time.time()
    for _ in range(int(seconds)):
        time.sleep(1)
        for i in range(N_IMU):
            q, fc, ok = read_q(i)
            print('t=%5.1f IMU%d fc=%d q=(%.4f %.4f %.4f %.4f) |q|=%.4f%s' % (time.time() - t0, i, fc, *q, np.linalg.norm(q), '' if ok else '  (no coherente)'))
    dt = time.time() - t0
    for i in range(N_IMU):
        da = (nav[i].read(NAV_SAMPLE_CNT) - c_acq0[i]) & 0xFFFFFFFF
        dc = (mad.read(MAD_FRAME_CNT + 4 * i) - c_core0[i]) & 0xFFFFFFFF
        print('IMU%d: adquiridas %d, procesadas %d (%.2f Hz), perdidas %d' % (i, da, dc, dc / dt, da - dc))
monitor(10)

7) Parar la adquisicion

In [ ]:
def stop():
    for i in range(N_IMU): nav[i].write(NAV_CTRL, 0)
# stop()